<a href="https://colab.research.google.com/github/CristianoMSilva/DistilBERT-FL-Playground/blob/main/04_federated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install Flower with the simulation engine (Ray)
!pip install -q "flwr[simulation]"

import flwr
print("Flower version:", flwr.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.6/283.6 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22

In [1]:
# Check the GPU of this session (recorded with the results)
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

GPU available: True
GPU: Tesla T4
GPU memory (GB): 15.6


In [2]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = load_dataset("zefang-liu/phishing-email-dataset")["train"].to_pandas()

# Clean (same steps as 01_eda.ipynb)
df = df.drop(columns=["Unnamed: 0"])
df = df.dropna(subset=["Email Text"])
conflicts = df.groupby("Email Text")["Email Type"].nunique()
df = df[~df["Email Text"].isin(conflicts[conflicts > 1].index)]
df = df.drop_duplicates(subset="Email Text")
word_count = df["Email Text"].str.split().str.len()
df = df[(word_count > 0) & (word_count < 1_000_000)].reset_index(drop=True)

# Labels: 1 = phishing, 0 = safe
df["label"] = (df["Email Type"] == "Phishing Email").astype(int)

# Stratified split: 80% train, 10% validation, 10% test (same as 03_baseline_training.ipynb)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(part)} emails, {part['label'].mean():.1%} phishing")

README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv: reconstructing file:   0%|          |  0.00B / 52.0MB            

Phishing_Email.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

Train: 14027 emails, 37.4% phishing
Validation: 1753 emails, 37.4% phishing
Test: 1754 emails, 37.4% phishing


In [3]:
from sklearn.model_selection import StratifiedKFold

NUM_CLIENTS = 3

# IID split: each client gets a random third of the training set, with the same phishing ratio
skf = StratifiedKFold(n_splits=NUM_CLIENTS, shuffle=True, random_state=42)
client_dfs = [train_df.iloc[idx] for _, idx in skf.split(train_df, train_df["label"])]

for i, part in enumerate(client_dfs):
    print(f"Client {i}: {len(part)} emails, {part['label'].mean():.1%} phishing")

Client 0: 4676 emails, 37.4% phishing
Client 1: 4676 emails, 37.4% phishing
Client 2: 4675 emails, 37.4% phishing


In [4]:
from datasets import Dataset
from transformers import AutoTokenizer

# Same settings as 03_baseline_training.ipynb (256 tokens)
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# DataFrame -> tokenized Hugging Face Dataset
def prepare(part):
    ds = Dataset.from_pandas(part[["Email Text", "label"]].reset_index(drop=True))
    return ds.map(lambda b: tokenizer(b["Email Text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["Email Text"])

# One tokenized dataset per client; validation and test stay with the server
client_ds = [prepare(part) for part in client_dfs]
val_ds = prepare(val_df)
test_ds = prepare(test_df)

for i, ds in enumerate(client_ds):
    print(f"Client {i}: {len(ds)} emails")
print(f"Server: {len(val_ds)} validation, {len(test_ds)} test")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4676 [00:00<?, ? examples/s]

Map:   0%|          | 0/4676 [00:00<?, ? examples/s]

Map:   0%|          | 0/4675 [00:00<?, ? examples/s]

Map:   0%|          | 0/1753 [00:00<?, ? examples/s]

Map:   0%|          | 0/1754 [00:00<?, ? examples/s]

Client 0: 4676 emails
Client 1: 4676 emails
Client 2: 4675 emails
Server: 1753 validation, 1754 test


In [5]:
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
collator = DataCollatorWithPadding(tokenizer)

# Fresh DistilBERT with a 2-class head (same as 03)
def get_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Local training on one client: class weights come only from that client's own labels
def train_local(model, ds, epochs=1, lr=2e-5):
    weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=np.array(ds["label"]))
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float, device=DEVICE))
    loader = DataLoader(ds, batch_size=16, shuffle=True, collate_fn=collator)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.to(DEVICE).train()
    for _ in range(epochs):
        for batch in loader:
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids=batch["input_ids"].to(DEVICE),
                               attention_mask=batch["attention_mask"].to(DEVICE)).logits
            loss = loss_fn(logits.float(), batch["labels"].to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

# Metrics with phishing (label 1) as the positive class
@torch.no_grad()
def evaluate(model, ds):
    loader = DataLoader(ds, batch_size=32, collate_fn=collator)
    model.to(DEVICE).eval()
    preds, labels = [], []
    for batch in loader:
        logits = model(input_ids=batch["input_ids"].to(DEVICE),
                       attention_mask=batch["attention_mask"].to(DEVICE)).logits
        preds += logits.argmax(-1).tolist()
        labels += batch["labels"].tolist()
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    return {"accuracy": accuracy_score(labels, preds), "precision": precision, "recall": recall,
            "f1": f1, "false_positive_rate": fp / (fp + tn)}

# Quick test: 300 emails from client 0, evaluated on 300 validation emails
model = get_model()
start = time.time()
train_local(model, client_ds[0].select(range(300)))
print(f"Training time: {time.time() - start:.0f} s")
print({k: round(v, 4) for k, v in evaluate(model, val_ds.select(range(300))).items()})

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training time: 5 s
{'accuracy': 0.9433, 'precision': 0.9423, 'recall': 0.8991, 'f1': 0.9202, 'false_positive_rate': np.float64(0.0314)}


In [7]:
from flwr.app import ArrayRecord, Context, Message, MetricRecord, RecordDict
from flwr.clientapp import ClientApp
from flwr.serverapp import Grid, ServerApp
from flwr.serverapp.strategy import FedAvg
from flwr.simulation import run_simulation
from transformers.utils import logging as hf_logging

NUM_ROUNDS = 5

# Fresh model without the UNEXPECTED/MISSING report (it would print every round)
def get_model():
    hf_logging.set_verbosity_error()
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Client: loads the global weights, trains 1 epoch on its own emails, sends the new weights back
client_app = ClientApp()

@client_app.train()
def client_train(msg: Message, context: Context):
    cid = context.node_config["partition-id"]
    model = get_model()
    model.load_state_dict(msg.content["arrays"].to_torch_state_dict())
    train_local(model, client_ds[cid])
    content = RecordDict({"arrays": ArrayRecord(model.state_dict()),
                          "metrics": MetricRecord({"num-examples": len(client_ds[cid])})})
    return Message(content=content, reply_to=msg)

# Server-side evaluation after every round; keeps the best round (highest validation F1)
history, best = [], {"f1": -1.0}

def server_evaluate(server_round: int, arrays: ArrayRecord):
    model = get_model()
    model.load_state_dict(arrays.to_torch_state_dict())
    metrics = {k: float(v) for k, v in evaluate(model, val_ds).items()}
    history.append({"round": server_round, **metrics})
    print(f"Round {server_round}: " + ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
    if metrics["f1"] > best["f1"]:
        best.update(f1=metrics["f1"], round=server_round, state=arrays.to_torch_state_dict())
    return MetricRecord(metrics)

# Server: FedAvg with all 3 clients training every round
server_app = ServerApp()

@server_app.main()
def server_main(grid: Grid, context: Context):
    strategy = FedAvg(fraction_train=1.0, fraction_evaluate=0.0, min_available_nodes=NUM_CLIENTS)
    strategy.start(grid=grid, initial_arrays=ArrayRecord(get_model().state_dict()),
                   num_rounds=NUM_ROUNDS, evaluate_fn=server_evaluate)

print("Ready")

Ready


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
